# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ShivanshRastogi315/FlyRank_Internship-repo/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Two paper findings + my methodology questions

**Methodology Audit of FlyRank Claims:**

1.  **Finding:** *"Long-form, comprehensive content acts as the primary driver for AI visibility."*
    *   **Where does the label come from?** The label is derived from trailing 90-day AI sessions, likely sourced from Google Search Console. 
    *   **Does the validation carry the claim?** No. The design proves *correlation*, not *causation*. Longer pages generally have higher domain authority and more backlinks. The validation design cannot definitively prove that adding 1,000 words to an arbitrary page will magically trigger an AI Overview.

2.  **Finding:** *"Updating stale content revives AI traffic capture."*
    *   **Where does the label come from?** The binary `is_ai_traffic` flag.
    *   **Does the validation carry the claim?** Not entirely. Our model evaluated a static 90-day snapshot. To truly prove "revival," the validation design needs a longitudinal A/B test (measuring the exact same page 30 days before an update vs 30 days after).

In [1]:
print("Methodology audit complete. No programmatic execution required for this section.")

Methodology audit complete. No programmatic execution required for this section.


## 2. My model under an honest split (before/after)

**The Danger of a Naive Split:**
If we use a standard random split (like we theoretically did in Week 5), pages from `Client A` end up in both the training and test sets. The model simply memorizes `Client A`'s overall domain authority, artificially inflating our precision (Data Leakage). 
By switching to an "Honest Split" (grouped by `client_id` as done in ML-08), we force the model to predict on entirely unseen businesses, simulating real-world agency onboarding. The performance drops, but the metric is finally honest.

In [2]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, GroupShuffleSplit
from sklearn.ensemble import RandomForestClassifier

df = pd.read_csv("../../data/raw/content_refresh_anonymized.csv")
df['is_ai_traffic'] = (df['ai_sessions_90d'] > 0).astype(int)
features = ['search_volume', 'avg_position', 'word_count', 'content_age_days', 'days_since_last_update', 'engagement_rate']
df[features] = df[features].fillna(0)

# --- 1. THE NAIVE SPLIT (Random 80/20) ---
X_train_n, X_test_n, y_train_n, y_test_n = train_test_split(df[features], df['is_ai_traffic'], test_size=0.2, random_state=42)
rf_naive = RandomForestClassifier(n_estimators=100, class_weight='balanced', random_state=42)
rf_naive.fit(X_train_n, y_train_n)

test_df_naive = df.loc[X_test_n.index].copy()
test_df_naive['naive_score'] = rf_naive.predict_proba(X_test_n)[:, 1]
naive_p20 = (test_df_naive.sort_values(by='naive_score', ascending=False).head(20)['ai_sessions_90d'] > 0).sum() / 20 * 100

# --- 2. THE HONEST SPLIT (Grouped by Client) ---
gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, test_idx = next(gss.split(df, groups=df['client_id']))
train_df, test_df = df.iloc[train_idx].copy(), df.iloc[test_idx].copy()

rf_honest = RandomForestClassifier(n_estimators=100, class_weight='balanced', random_state=42)
rf_honest.fit(train_df[features], train_df['is_ai_traffic'])
test_df['honest_score'] = rf_honest.predict_proba(test_df[features])[:, 1]
honest_p20 = (test_df.sort_values(by='honest_score', ascending=False).head(20)['ai_sessions_90d'] > 0).sum() / 20 * 100

print("--- MODEL SPLIT VALIDATION ---")
print(f"Naive Split Precision@20 (Cheating via Domain Leakage): {naive_p20:.2f}%")
print(f"Honest Split Precision@20 (Real-World Generalization):  {honest_p20:.2f}%")

--- MODEL SPLIT VALIDATION ---
Naive Split Precision@20 (Cheating via Domain Leakage): 15.00%
Honest Split Precision@20 (Real-World Generalization):  10.00%


## 3. Leakage audit

**Final Feature Sanity Check:**
We are conducting one last correlation check on our final feature vector. We must ensure no target-derived metrics (like overall clicks or post-facto trend directions) have snuck into the `features` array before final deployment.

In [3]:
# Create a temporary dataframe to check correlations against the target
leakage_check = df[features].copy()
leakage_check['TARGET'] = df['is_ai_traffic']

# Calculate correlation matrix and extract the target column
correlations = leakage_check.corr()['TARGET'].sort_values(ascending=False).drop('TARGET')

print("--- FINAL LEAKAGE CORRELATION CHECK ---")
print(correlations)
print("\nVerdict: Clean. No correlations exceed a suspicious 0.6 threshold. No future-looking metrics present.")

--- FINAL LEAKAGE CORRELATION CHECK ---
word_count                0.234981
days_since_last_update    0.137362
avg_position              0.026244
engagement_rate           0.024245
search_volume            -0.016341
content_age_days         -0.024388
Name: TARGET, dtype: float64

Verdict: Clean. No correlations exceed a suspicious 0.6 threshold. No future-looking metrics present.


## 4. Claim rewrite

*   **Original Bold Claim:** *"Our Machine Learning algorithm predicts exactly which pages will rank in AI Overviews, allowing us to reverse-engineer the search engine's algorithm."*
*   **Rewritten Safe Claim:** *"Our Random Forest pipeline identifies observed structural features—such as word count depth and content freshness—that directionally correlate with historical AI traffic, serving as a data-driven decision-support tool for content prioritization."*

In [4]:
print("Claims safely rewritten to align with strict data science ethics. The project is ready for submission.")

Claims safely rewritten to align with strict data science ethics. The project is ready for submission.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.